# 09 · The `okf` package

Stage 1 of the backend. Notebook `07_okf_hybrid_query` won the comparison, but a notebook can't be imported by a server. So the same pipeline now lives in a Python package, `okf/`, that every later stage builds on:

```
                      ┌─────────────── okf.OKF ───────────────┐
question ──► Retriever: BM25 + Qdrant + links ──► context ──► LLM ──► Answer (pydantic)
                      └───────────────────────────────────────┘
                              ▲            ▲            ▲
 stage 2  MCP server ─────────┘ (tools: search_okf, ask_okf, get_concept)
 stage 4  OKF agent (A2A) ──── calls the MCP tools
 stage 6  LitServe ─────────── serves the agents
```

**Nothing about retrieval changed.** Same settings, same BM25, same Qdrant query, same RRF, same context, same system prompt. This notebook proves it against the answers you saved for the evaluation.

## Where each piece of `07` went

| `okf/` module | from `07` | what it does |
|---|---|---|
| `config.py` | the settings cell | `Settings`: every knob with the evaluated default. Override with `OKF_<NAME>` in `.env` (e.g. `OKF_N_SECTIONS=4`) or `Settings.load(n_sections=4)`. |
| `bundle.py` | `load_bundle`, links, `render`, `topic_text` | `Bundle`: concepts, `out_links` / `in_links`, rendering a context block |
| `chunkmap.py` | step 3 | chunk → section map, **cached** in `artifacts/cache/` so a server starts fast |
| `search.py` | `tokenize`, `BM25`, `rrf`, `split_question`, `papers_named` | keyword search and query helpers |
| `vectors.py` | `qdrant_search`, step 4 upload | `VectorStore`: Qdrant search, plus `sync_topics()` to refresh `okf_topics` when the bundle changes |
| `retrieve.py` | `retrieve`, `build_context` | `Retriever` |
| `llm.py` | `SYSTEM_PROMPT`, `ask_llm` | `OllamaLLM` (local) and `OpenAICompatLLM` (Mercury, OpenRouter… for the public version) |
| `schemas.py` | (new) | pydantic models: `Answer`, `Source`, `Passage`, `Retrieval` |
| `pipeline.py` | `ask`, `sources_for` | `OKF`: `ask()`, `search()`, `get_concept()`, `status()` |

**Why pydantic models instead of dicts?** The later stages all need to know the *shape* of an answer. `Answer.model_json_schema()` gives the MCP server its tool output schema, the A2A agent card its description, and Outlines its target. A dict can't do that, and a typo in a key would only show up at runtime.

**One small change: links are visited in sorted order.** Python sets have no fixed order between runs, and `07` looped over link sets. When two topic pages tied, the order changed from run to run (in a test with fake vector results, 2–5 of 33 questions changed with the hash seed). The package sorts them, so the same question always gives the same context.

## 1. Load

`OKF.load()` finds the project folder, reads `.env`, loads the bundle, connects to Qdrant and checks the model. The first load builds the chunk → section cache (a few seconds); after that it's read from disk.

`%autoreload 2` re-imports the package when you edit a module, so you can change `okf/*.py` and re-run a cell without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import time
from pathlib import Path

import pandas as pd

from okf import OKF, Answer, Settings

start = time.perf_counter()
okf = OKF.load()
print(f"loaded in {time.perf_counter() - start:.1f} s")
for k, v in okf.status().items():
    print(f"{k:15} {v}")

## 2. Retrieval only

`retrieve()` is the same as in `07`: topic pages, then sections, each with who found it (`vector`, `bm25`, `link`). `search()` goes one step further and returns the context blocks the model would see, each with its `Source`. That's exactly what the MCP `search_okf` tool will return in stage 2.

In [ ]:
q = "What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?"
r = okf.retrieve(q)
print("parts:", r.parts, "| papers named:", r.papers or "-", "| vectors:", r.vectors_used)
for t in r.topics:
    print(f"   topic   {t}")
for h in r.sections:
    print(f"   section {h.found_by:18} {h.score:5.1f}  {h.id}")
print("timing:", r.timing)

print("\nwhat search() returns:")
for p in okf.search(q):
    print(f"   [{p.source.id}]  {p.source.label}  ({len(p.text)} chars)")

## 3. Check: same context as the evaluation (no LLM, ~1 minute)

For each of the 30 eval questions, build the context with the package and compare it block by block with `retrieval_context` in `artifacts/results/okf_hybrid.jsonl`, the exact blocks `07` gave the model during the evaluation.

What to expect:
- **identical** for nearly all questions.
- **same blocks, different order**: the tie-order fix above. Harmless.
- **different blocks**: look at those. Usual causes: the bundle changed since the eval, Qdrant returned something different, or vectors are off (check `status` above).

In [ ]:
assert okf.retriever.vectors_available, "vectors are off: this check compares against the hybrid run, so connect Qdrant first (see status)"
saved = [json.loads(line) for line in (okf.settings.project_root / "artifacts/results/okf_hybrid.jsonl").open(encoding="utf-8") if line.strip()]

rows = []
for item in saved:
    _, used, blocks = okf.retriever.build_context(okf.retrieve(item["question"]))
    old = item["retrieval_context"]
    verdict = ("identical" if blocks == old else
               "same blocks, other order" if sorted(blocks) == sorted(old) else "DIFFERENT")
    head = lambda bs: [b.split("\n", 1)[0].split("]")[0].strip("# [") for b in bs]
    rows.append({"id": item["id"], "verdict": verdict, "only_saved": sorted(set(head(old)) - set(head(blocks))),
                 "only_package": sorted(set(head(blocks)) - set(head(old)))})
check = pd.DataFrame(rows)
print(check.verdict.value_counts().to_string())
display(check[check.verdict != "identical"])

## 4. Check: same answers (uses the LLM, ~1 minute)

Same context + same prompt + temperature 0 should give the same answer. Ollama isn't bit-for-bit deterministic on a GPU, so a word or two can change; `similarity` (0–1, by `difflib`) shows how close each answer is to the saved one.

In [ ]:
import difflib

rows = []
for item in saved[:5]:
    a = okf.ask(item["question"])
    rows.append({"id": item["id"], "similarity": round(difflib.SequenceMatcher(None, a.answer, item["answer"]).ratio(), 3),
                 "same_context": a.retrieval_context == item["retrieval_context"],
                 "generation_ms": a.generation_ms, "answer": a.answer[:90]})
display(pd.DataFrame(rows))

## 5. What an `Answer` looks like

This is the object every later stage passes around. `sources` holds only the blocks the answer actually **cites** (and that were really in the context). Citations to ids that weren't in the context are dropped here; stage 3 turns them into a guardrail warning.

In [ ]:
a = okf.ask("What score did GPT-4 get on the bar exam?")
print(a.answer, "\n")
for s in a.sources:
    print(f"  [{s.id}] {s.label}")
print(f"\nrefused: {a.refused} | model: {a.model} | retrieval {a.retrieval_ms:.0f} ms | generation {a.generation_ms:.0f} ms")

print("\nas JSON (first 600 characters):")
print(a.model_dump_json(indent=2)[:600], "...")
print("\nfields in the schema:", list(Answer.model_json_schema()["properties"]))

## 6. Swapping the answering model (for the public version later)

Locally the answering model is qwen 4B on Ollama. A public server usually has no GPU, so `llm.py` also speaks the OpenAI-style API that Mercury, OpenRouter and most providers use. Nothing else changes: same retrieval, same prompt.

To switch everything (MCP server, agents, UI), put this in `.env`:
```
OKF_LLM_PROVIDER=openai
OKF_LLM_BASE_URL=https://api.inceptionlabs.ai/v1
OKF_LLM_MODEL=mercury-2.5
OKF_LLM_API_KEY=...            # the same key as INCEPTION_API_KEY
```
Or try it on one question below. Mercury's answers haven't been evaluated: before switching for real, re-run the save-results step and `08` with it.

In [ ]:
TRY_MERCURY = False      # True to compare one answer (uses ~5k Mercury tokens)

if TRY_MERCURY:
    import os
    from okf.llm import make_llm
    mercury_settings = okf.settings.with_(llm_provider="openai", llm_base_url="https://api.inceptionlabs.ai/v1",
                                          llm_model="mercury-2.5", llm_api_key=os.getenv("INCEPTION_API_KEY", ""))
    mercury = OKF(mercury_settings, okf.bundle, okf.retriever, make_llm(mercury_settings))   # same retrieval, other model
    m = mercury.ask("What score did GPT-4 get on the bar exam?")
    print(m.answer, "\n\nsources:", [s.id for s in m.sources], f"| {m.generation_ms:.0f} ms")

## 7. Outside the notebook

Install the package into the `unsloth` env once, in **editable** mode (`-e`: the env points at this folder, so edits take effect without reinstalling):
```
conda activate unsloth
cd C:\Users\Swapn\OneDrive\Documents\GitHub\OKF
pip install -e .
```
Then from any terminal in the project folder:
```
python -m okf --status
python -m okf "How does HELM measure calibration?"
python -m okf --search "bar exam"          # retrieval only
python -m okf --no-vectors "..."           # keyword search + links only
```
The MCP server (stage 2), agents and LitServe run as their own processes, so they need this install.

**Next: stage 2, the MCP server.** It wraps `okf.search`, `okf.ask` and `okf.get_concept` as tools you can call from Claude Desktop or the MCP Inspector before any agent exists.